# Qwen3-Embedding-4B for new postings, on a Kaggle GPU

Embeds the chunks in `pending.json.gz`, written by `make export-chunk-vectors-pending`, with the whole-index run's settings (left padding, last-token pooling, unit length) so the new vectors compare with the stored ones. One change: **float16**, because Kaggle's T4 and P100 have no native bfloat16. The export puts postings that already have vectors first, and `make import-chunk-vectors` compares them with the stored ones before it writes anything.

1. **Settings: Accelerator GPU (T4 or P100), Internet on.** A GPU still holding another model (the 27B chat model) has no room: restart the session first.
2. **Add `pending.json.gz`**: as a private dataset (Add Input), or uploaded to `/kaggle/working/`. Cell 2 finds it.
3. **Run All.** About 2 minutes to download the model, then roughly 20 chunks a second on one T4.
4. **Cell 6 saves `qwen4b_batch_f16.npy` in `/kaggle/working/`.** Download it into the export's folder on the Mac and run the import command the export printed.

Only public posting text is in this notebook.

## 1. Check the session has a GPU and internet

In [ ]:
import importlib.metadata as md
import shutil
import subprocess
import sys
import urllib.request

facts = {"python": sys.version.split()[0]}
for name in ("torch", "transformers", "numpy"):
    try:
        facts[name] = md.version(name)
    except Exception:
        facts[name] = None
if shutil.which("nvidia-smi"):
    facts["gpus"] = (
        subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
            capture_output=True,
            text=True,
        )
        .stdout.strip()
        .splitlines()
    )
else:
    facts["gpus"] = []
try:
    urllib.request.urlopen("https://huggingface.co/api/models/Qwen/Qwen3-Embedding-4B", timeout=10)
    facts["internet"] = True
except Exception as exc:
    facts["internet"] = type(exc).__name__
print(facts)
if not facts["gpus"]:
    raise RuntimeError("No GPU: Settings -> Accelerator -> GPU, then restart the session.")
if facts["internet"] is not True:
    raise RuntimeError("No internet: Settings -> Internet on, so the model can be downloaded.")

## 2. Read pending.json.gz

In [ ]:
import glob
import gzip
import json
import os

NAME = "pending.json.gz"


def candidates():
    """Every copy in /kaggle/working or an attached dataset, newest first."""
    found = set(glob.glob("/kaggle/working/" + NAME)) | set(
        glob.glob("/kaggle/input/**/" + NAME, recursive=True)
    )
    existing = [path for path in found if os.path.isfile(path)]
    return sorted(existing, key=os.path.getmtime, reverse=True)


hits = candidates()
if not hits:
    raise FileNotFoundError(
        f"{NAME} is not in /kaggle/working or an attached dataset. Add it and run this again."
    )
src = hits[0]
if len(hits) > 1:
    print("several copies; using the newest:", src)
with gzip.open(src, "rt", encoding="utf-8") as handle:
    data = json.load(handle)
chunks = [text for posting in data["postings"] for text in posting["chunks"]]
with open("/tmp/batch_chunks.json", "w") as handle:
    json.dump({"chunks": chunks}, handle)
print(
    f"reading {src}: {len(data['postings']):,} postings, {len(chunks):,} chunks, "
    f"made {data.get('made')}"
)

## 3. Make sure transformers knows Qwen3

In [ ]:
import importlib.metadata as md
import subprocess
import sys

major, minor = (int(part) for part in md.version("transformers").split(".")[:2])
if (major, minor) < (4, 51):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51"], check=True)
    print("upgraded transformers; restart the session if the next cell fails to import it")
print("transformers", md.version("transformers"), "| torch", md.version("torch"))

## 4. The embedding script

In [ ]:
%%writefile /tmp/gpu_batch.py
import json, time
import numpy as np
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

# The whole-index run's model and settings (2026-11-02), except precision:
# float16 here, bfloat16 there. The import's check measures what that costs.
MODEL = "Qwen/Qwen3-Embedding-4B"
# Tokens per batch at every length bucket, so a 16 GB T4 holds the model and a batch.
TOKENS_PER_BATCH = 8192
PROGRESS_EVERY = 0.1
OUT = "/kaggle/working/qwen4b_batch_f16.npy"

chunks = json.load(open("/tmp/batch_chunks.json"))["chunks"]
dev = torch.device("cuda:0")
dtype_kw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")


def say(msg):
    print(msg, flush=True)


def bucket(n):
    return 128 if n <= 128 else 256 if n <= 256 else 512


def load(dtype):
    model = AutoModel.from_pretrained(MODEL, attn_implementation="sdpa", **{dtype_kw: dtype})
    return model.to(dev).eval()


def embed(model, texts, lens, progress=False):
    order = sorted(range(len(texts)), key=lambda i: lens[i])
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float16)
    i, done, next_report, t0 = 0, 0, PROGRESS_EVERY, time.perf_counter()
    while i < len(order):
        size = bucket(lens[order[i]])
        per_batch = TOKENS_PER_BATCH // size
        idx = []
        while i < len(order) and len(idx) < per_batch and bucket(lens[order[i]]) == size:
            idx.append(order[i]); i += 1
        enc = tok([texts[j] for j in idx], padding="max_length", truncation=True, max_length=size,
                  return_tensors="pt")
        with torch.no_grad():
            hidden = model(input_ids=enc["input_ids"].to(dev),
                           attention_mask=enc["attention_mask"].to(dev)).last_hidden_state
            vec = torch.nn.functional.normalize(hidden[:, -1, :].float(), p=2, dim=1)
        out[idx] = vec.cpu().numpy().astype(np.float16)
        done += len(idx)
        if progress and done / len(texts) >= next_report:
            spent = time.perf_counter() - t0
            say(f"  {done:,} of {len(texts):,} chunks ({done / len(texts):.0%}) in {spent / 60:.1f} min")
            next_report += PROGRESS_EVERY
    return out


lengths = [len(x) for x in tok(chunks, truncation=True, max_length=512)["input_ids"]]
warm = {}
for i, n in enumerate(lengths):
    warm.setdefault(bucket(n), i)
result = {"chunks": len(chunks), "tokens": sum(lengths), "gpu": torch.cuda.get_device_name(0)}
# float16 first. If it overflows on the warm-up chunks, bfloat16 where torch can run it.
for dtype in (torch.float16, torch.bfloat16):
    if dtype is torch.bfloat16 and not torch.cuda.is_bf16_supported():
        continue
    t = time.perf_counter()
    model = load(dtype)
    sample = embed(model, [chunks[i] for i in warm.values()], [lengths[i] for i in warm.values()])
    if np.isfinite(sample).all():
        result.update(dtype=str(dtype), load_s=round(time.perf_counter() - t))
        say(f"ready: {dtype}, model loaded in {result['load_s']} s")
        break
    say(f"{dtype} gave non-finite vectors on the warm-up chunks; trying the next precision")
    del model
    torch.cuda.empty_cache()
else:
    print("RESULT " + json.dumps({**result, "error": "no precision gave finite vectors"}), flush=True)
    raise SystemExit(1)

torch.cuda.synchronize()
t = time.perf_counter()
docs = embed(model, chunks, lengths, progress=True)
torch.cuda.synchronize()
result.update(encode_s=round(time.perf_counter() - t), finite=bool(np.isfinite(docs).all()),
              shape=list(docs.shape))
np.save(OUT, docs)
print("RESULT " + json.dumps(result), flush=True)

## 5. Embed every chunk

In [ ]:
import json
import subprocess
import sys

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
proc = subprocess.Popen(
    [sys.executable, "/tmp/gpu_batch.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
result = None
for line in proc.stdout:
    if line.startswith("RESULT "):
        result = json.loads(line[7:])
    else:
        print(line, end="")
proc.wait()
ok = bool(result and result.get("finite") and result.get("shape", [0])[0] == len(chunks))
print(
    f"embedded {result['chunks']:,} chunks in {result['encode_s'] / 60:.1f} min on "
    f"{result['gpu']} ({result['dtype']})"
    if ok
    else f"failed: {result}"
)

## 6. Where the vectors are

In [ ]:
if not ok:
    raise RuntimeError("The embedding did not finish cleanly (cell 5); nothing was saved.")
path = "/kaggle/working/qwen4b_batch_f16.npy"
print(f"{path} ({os.path.getsize(path) / 1e6:.1f} MB) is ready.")
print("Download it into the export's folder on the Mac, then run the import it printed.")